In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Parameters (must match the simulation)
n_sys = 10

results_file = f"../results/vdp_errors_sys{n_sys}.npz"
try:
    data = np.load(results_file)
except FileNotFoundError:
    print(f"Error: {results_file} not found. Please run the simulation first.")
    raise

errors_gEDMD        = data["errors_gEDMD"]
errors_iEDMD        = data["errors_iEDMD"]
errors_pEDMD        = data["errors_pEDMD"]
errors_e_pEDMD      = data["errors_e_pEDMD"]
errors_sparse_iEDMD = data["errors_sparse_iEDMD"]
cond_gEDMD        = data["condn_gEDMD"]
cond_iEDMD        = data["condn_iEDMD"]
cond_pEDMD        = data["condn_pEDMD"]
cond_e_pEDMD      = data["condn_e_pEDMD"]
n_obs_pike       = int(data["n_obs_pike"])
n_obs_classic = int(data["n_obs_classic"])
n_train     = int(data["n_train"])
n_test      = int(data["n_test"])

print(f"Loaded {results_file}  —  shape: {errors_gEDMD.shape}")
print(f"  n_obs_pike={n_obs_pike}, n_obs_classic={n_obs_classic}, n_train={n_train}, n_test={n_test}")

In [ ]:
points_range = np.arange(1, n_train + 1)

# Log-spaced marker positions (only where data is not NaN)
n_markers = 8
x_markers = np.unique(np.logspace(0, np.log10(n_train), num=n_markers, dtype=int))

fig, ax = plt.subplots()

# Vertical lines for dictionary sizes
ax.axvline(n_obs_classic, color='black', linestyle='--', linewidth=0.8)
ax.axvline(n_obs_pike, color='black', linestyle='--', linewidth=0.8)

# gEDMD
ax.loglog(points_range, errors_gEDMD, color="C0", linewidth=1)
ym = np.interp(x_markers, points_range, errors_gEDMD)
ax.loglog(x_markers, ym, 'o', color="C0", markersize=5, clip_on=False)
ax.plot([], [], f'-o', color="C0", label="gEDMD")

# pEDMD
ax.loglog(points_range, errors_pEDMD, color="C1", linewidth=1)
ym = np.interp(x_markers, points_range, errors_pEDMD)
ax.loglog(x_markers, ym, 's', color="C1", markersize=5, clip_on=False)
ax.plot([], [], f'-s', color="C1", label="pEDMD")

# iEDMD
ax.loglog(points_range, errors_iEDMD, color="C2", linewidth=1)
ym = np.interp(x_markers, points_range, errors_iEDMD)
ax.loglog(x_markers, ym, '^', color="C2", markersize=5, clip_on=False)
ax.plot([], [], f'-^', color="C2", label="iEDMD")

# e-pEDMD
ax.loglog(points_range, errors_e_pEDMD, color="C3", linewidth=1)
ym = np.interp(x_markers, points_range, errors_e_pEDMD)
ax.loglog(x_markers, ym, 'D', color="C3", markersize=5, clip_on=False)
ax.plot([], [], f'-D', color="C3", label="e-pEDMD")

# sparse iEDMD
ax.loglog(points_range, errors_sparse_iEDMD, color="C4", linewidth=1)
ym = np.interp(x_markers, points_range, errors_sparse_iEDMD)
ax.loglog(x_markers, ym, 'p', color="C4", markersize=5, clip_on=False)
ax.plot([], [], f'-p', color="C4", label="s-iEDMD")

ax.set_xlabel('Number of training points $L$')
ax.set_ylabel(r'Average prediction error $\left\|\dot{\boldsymbol{x}} - \widehat{\dot{\boldsymbol{x}}}\right\|_2$')
ax.legend(ncol=2)
ax.grid()

for spine in ax.spines.values():
    spine.set_zorder(1)

fig.savefig(f"../results/vdp_error_vs_L.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Condition number plot
fig, ax = plt.subplots()
for cond, marker, color, label in [
    (cond_gEDMD, 'o', 'C0', 'gEDMD'),
    (cond_iEDMD, 's', 'C1', 'iEDMD'),
    (cond_pEDMD, 'D', 'C2', 'pEDMD'),
    (cond_e_pEDMD, 'v', 'C3', 'e-pEDMD'),
]:
    valid = ~np.isnan(cond)
    x_valid = points_range[valid]
    y_valid = cond[valid]
    ax.loglog(x_valid, y_valid, f'-', color=color, label=label)

ax.axvline(n_obs_classic, color='black', linestyle='--', linewidth=0.8)
ax.axvline(n_obs_pike, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('Number of training points $L$')
ax.set_ylabel('Condition number')
ax.legend()
ax.grid(True, which='both', linewidth=0.3)
plt.show()

In [ ]:
data = np.load("../results/vdp_sparse_matrices.npz")
K_c, K_mask = data["K_c"], data["K_mask"]

nrows, ncols = K_c.shape

# --- Sizing ---
cell_size = 0.03  # inches per matrix cell — tune to taste
cbar_width = 0.3  # inches for each colorbar
gap = 0.1         # inches between the two panels
margin_x = 0.15
margin_y = 0.15

panel_w = ncols * cell_size
panel_h = nrows * cell_size
fig_w = 2 * panel_w + 2 * cbar_width + gap + 2 * margin_x
fig_h = panel_h + 2 * margin_y

fig = plt.figure(figsize=(fig_w, fig_h))

# Normalised positions
def add_panel(fig, left_in, matrix, cmap, cbar_label, vmin=None, vmax=None):
    ax_l = left_in / fig_w
    ax_b = margin_y / fig_h
    ax_w = panel_w / fig_w
    ax_h = panel_h / fig_h
    ax = fig.add_axes([ax_l, ax_b, ax_w, ax_h])
    im = ax.imshow(matrix, cmap=cmap, aspect='equal', vmin=vmin, vmax=vmax)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.set_title(cbar_label, fontsize=10)
    # colorbar right next to the panel
    cb_l = (left_in + panel_w + 0.05) / fig_w
    cb_w = 0.08 / fig_w
    cax = fig.add_axes([cb_l, ax_b, cb_w, ax_h])
    fig.colorbar(im, cax=cax)
    cax.tick_params()
    return ax

left1 = margin_x
left2 = margin_x + panel_w + cbar_width + gap

vlim = np.max(np.abs(K_c))
add_panel(fig, left1, K_mask, 'binary', r"$\overline{\boldsymbol{K}}$")
add_panel(fig, left2, K_c, 'seismic', r"$\boldsymbol{K}_c$", vmin=-vlim, vmax=vlim)

plt.savefig("../results/vdp_sparse_matrices.pdf", bbox_inches='tight', pad_inches=0.02)
plt.show()

In [ ]:
# Eigenvalues plot
L = 1000
data = np.load(f"../results/vdp_K_matrices_L{L}.npz")

# Get the matrices
K_gEDMD = data["K_gEDMD"]
K_iEDMD = data["K_iEDMD"]
K_pEDMD = data["K_pEDMD"]
K_e_pEDMD = data["K_e_pEDMD"]
K_sparse_iEDMD = data["K_sparse_iEDMD"]

# Compute eigenvalues
eig_gEDMD = np.linalg.eigvals(K_gEDMD)
eig_iEDMD = np.linalg.eigvals(K_iEDMD)
eig_pEDMD = np.linalg.eigvals(K_pEDMD)
eig_e_pEDMD = np.linalg.eigvals(K_e_pEDMD)
eig_sparse_iEDMD = np.linalg.eigvals(K_sparse_iEDMD)

fig, ax = plt.subplots()
ax.axvline(0, color='black', linestyle='-', linewidth=0.8)
ax.plot(eig_gEDMD.real, eig_gEDMD.imag, 'o', color='C0', label='gEDMD')
ax.plot(eig_iEDMD.real, eig_iEDMD.imag, 's', color='C1', label='iEDMD')
ax.plot(eig_pEDMD.real, eig_pEDMD.imag, 'D', color='C2', label='pEDMD')
ax.plot(eig_e_pEDMD.real, eig_e_pEDMD.imag, 'v', color='C3', label='e-pEDMD')
ax.plot(eig_sparse_iEDMD.real, eig_sparse_iEDMD.imag, 'p', color='C4', label='s-iEDMD')
ax.set_xlabel('Real part')
ax.set_ylabel('Imaginary part')
ax.legend(loc='best', ncol=2)
ax.grid()
plt.savefig(f"../results/vdp_eigenvalues_L{L}.pdf", bbox_inches='tight')
plt.show()

In [ ]:
# Trajectories plot
L = 1000
data = np.load(f"../results/vdp_trajectories_L{L}.npz")

# Get the trajectories
true_traj = data["true_traj"]
pred_traj_gEDMD = data["pred_traj_gEDMD"]
pred_traj_iEDMD = data["pred_traj_iEDMD"]
pred_traj_pEDMD = data["pred_traj_pEDMD"]
pred_traj_e_pEDMD = data["pred_traj_e_pEDMD"]
pred_traj_sparse_iEDMD = data["pred_traj_sparse_iEDMD"]
t_eval = data["t_eval"]

# Error vs time plot
fig, ax = plt.subplots()
error_gEDMD = np.linalg.norm(true_traj - pred_traj_gEDMD, axis=0) / np.linalg.norm(true_traj, axis=0)
error_iEDMD = np.linalg.norm(true_traj - pred_traj_iEDMD, axis=0) / np.linalg.norm(true_traj, axis=0)
error_pEDMD = np.linalg.norm(true_traj - pred_traj_pEDMD, axis=0) / np.linalg.norm(true_traj, axis=0)
error_e_pEDMD = np.linalg.norm(true_traj - pred_traj_e_pEDMD, axis=0) / np.linalg.norm(true_traj, axis=0)
error_sparse_iEDMD = np.linalg.norm(true_traj - pred_traj_sparse_iEDMD, axis=0) / np.linalg.norm(true_traj, axis=0)
ax.semilogy(t_eval, error_gEDMD, 'C0-o', label='gEDMD', markevery=100)
ax.semilogy(t_eval, error_iEDMD, 'C1-^', label='iEDMD', markevery=100)
ax.semilogy(t_eval, error_pEDMD, 'C2-s', label='pEDMD', markevery=100)
ax.semilogy(t_eval, error_e_pEDMD, 'C3-D', label='e-pEDMD', markevery=100)
ax.semilogy(t_eval, error_sparse_iEDMD, 'C4-p', label='s-iEDMD', markevery=100)
ax.set_xlabel('Time')
ax.set_ylabel('Trajectory relative error')
ax.legend(ncol=2)
ax.grid()
plt.savefig(f"../results/vdp_error_vs_time_L{L}.pdf", bbox_inches='tight')
plt.show()